# 01 Data understanding (PEMS-BAY)

Run the sections in order. **Part A** inspects the raw files (Day 2). **Part B** checks the hourly table
that `src/01_build_dataset.py` produced (Day 5). Everything saved to `results/data_quality/` feeds the
website's Data page. Write your own one-line conclusion under each check.

In [ ]:
import json, os, sys
from pathlib import Path

# Work from the repository root whether the notebook is opened from notebooks/ or the root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

cfg = json.load(open("configs/dataset.json"))
OUT = Path("results/data_quality"); OUT.mkdir(parents=True, exist_ok=True)
cfg

## Part A: raw files
### A1. HDF5 keys, shape, time range

In [ ]:
from load_raw import list_keys, load_speed_wide, load_speed_table, read_sensor_locations
from hourly import to_hourly_wide

keys = list_keys(cfg["speed_file"])
print("HDF5 keys:", keys)
key = cfg["speed_key"] or keys[0]
raw = pd.read_hdf(cfg["speed_file"], key=key)
print("shape (timestamps x sensors):", raw.shape)
print("time range:", raw.index.min(), "->", raw.index.max())
print("step between rows:\n", raw.index.to_series().diff().value_counts().head())
raw.iloc[:5, :5]

### A2. How are missing values encoded? (zeros in long blocks = outages)

In [ ]:
zero_share = (raw == 0).mean()
print("Share of exact-zero readings overall:", round(float((raw == 0).values.mean()), 5))
print("Share of NaN readings overall:", round(float(raw.isna().values.mean()), 5))

def longest_run(mask: pd.Series) -> int:
    runs = mask.ne(mask.shift()).cumsum()
    return int(mask.groupby(runs).sum().max())

longest_zero = raw.apply(lambda c: longest_run(c == 0))
print("Sensors with a zero run longer than 12 steps (1 hour):", int((longest_zero > 12).sum()))
longest_zero.sort_values(ascending=False).head(10)

If zeros appear in long blocks they are outages, and `load_raw.py` already turns them into NaN. If there are almost none (expected for PEMS-BAY), say so here.

### A3. Sensor location file

In [ ]:
locs = read_sensor_locations(cfg["sensor_locations_file"])
print(locs.shape)
print("sensors in the speed file but not in the location file:",
      len(set(map(str, raw.columns)) - set(locs["sensor_id"])))
print("lat range:", locs["latitude"].min(), locs["latitude"].max(),
      "| lon range:", locs["longitude"].min(), locs["longitude"].max())
locs.head()

### A4. Long format and units (mph to km/h)

In [ ]:
sample_ids = [str(c) for c in raw.columns[:3]]
sample = load_speed_table(cfg["speed_file"], cfg["speed_key"], sensors=sample_ids)
display(sample.head())
print(sample.groupby("segment_id")["speed_kmh"].describe().round(1))
print("Check: raw max (mph) * 1.609344 =", round(float(raw[sample_ids].max().max()) * 1.609344, 1), "km/h")

### A5. Hourly build and how many hours were discarded

In [ ]:
wide = load_speed_wide(cfg["speed_file"], cfg["speed_key"])
hourly_wide, stats = to_hourly_wide(wide, cfg["min_readings_per_hour"])
print(stats)
print("Share of sensor-hours discarded by the 9-of-12 rule:",
      round(stats["hours_partial_discarded"] / stats["hours_total"], 5))

## Part B: data-quality checks on `data/processed/hourly.parquet` (Guide 7.9)

In [ ]:
hourly = pd.read_parquet("data/processed/hourly.parquet")
print(hourly.shape, "| segments:", hourly["segment_id"].nunique())
hourly.head()

### B1. Time range and count per sensor

In [ ]:
rng = hourly.groupby("segment_id")["timestamp"].agg(["min", "max", "count"])
rng.to_csv(OUT / "time_range.csv")
rng

### B2. Missing rate per sensor and per month

In [ ]:
miss_sensor = hourly.groupby("segment_id")["speed_kmh"].apply(lambda s: s.isna().mean()).rename("missing_rate")
miss_sensor.to_csv(OUT / "missing_per_sensor.csv")
hourly["month"] = hourly["timestamp"].dt.to_period("M").astype(str)
miss_month = hourly.groupby("month")["speed_kmh"].apply(lambda s: s.isna().mean()).rename("missing_rate")
miss_month.to_csv(OUT / "missing_per_month.csv")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
miss_sensor.plot.bar(ax=ax[0], title="Missing hours per sensor"); ax[0].set_ylabel("share missing")
miss_month.plot.bar(ax=ax[1], title="Missing hours per month"); ax[1].set_ylabel("share missing")
plt.tight_layout(); plt.savefig(OUT / "missing_rates.png", dpi=150); plt.show()

### B3. Flat runs longer than 6 hours (stuck sensors)

In [ ]:
def longest_flat_run(s: pd.Series) -> int:
    s = s.dropna()
    if s.empty:
        return 0
    groups = s.ne(s.shift()).cumsum()
    return int(s.groupby(groups).size().max())

flat = hourly.sort_values("timestamp").groupby("segment_id")["speed_kmh"].apply(longest_flat_run).rename("longest_flat_run_h")
flat.to_csv(OUT / "flat_runs.csv")
print("Sensors with a flat run over 6 hours:", int((flat > 6).sum()))
flat

### B4. Duplicates

In [ ]:
dups = int(hourly.duplicated(["timestamp", "segment_id"]).sum())
print("Duplicated (timestamp, segment_id) rows:", dups)
assert dups == 0

### B5. Timezone check: do the slow hours match known rush hours?

In [ ]:
by_hour = hourly.groupby(hourly["timestamp"].dt.hour)["speed_kmh"].mean()
fig, ax = plt.subplots(figsize=(7, 3.5))
by_hour.plot(ax=ax, marker="o"); ax.set_xlabel("hour of day"); ax.set_ylabel("mean speed (km/h)")
ax.set_title("Mean speed by hour (expect dips around 7-9 and 16-18)")
plt.tight_layout(); plt.savefig(OUT / "speed_by_hour.png", dpi=150); plt.show()
print("Slowest three hours:", by_hour.nsmallest(3).index.tolist())

### B6. Speed distribution per sensor

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
hourly["speed_kmh"].plot.hist(bins=50, ax=ax[0], title="All segments: hourly speed (km/h)")
sns.boxplot(data=hourly, x="segment_id", y="speed_kmh", ax=ax[1])
ax[1].set_title("Speed by segment"); ax[1].tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.savefig(OUT / "speed_distribution.png", dpi=150); plt.show()
hourly["speed_kmh"].describe().round(1)

### B7. Speed by hour and weekday (heatmap)

In [ ]:
grid = (hourly.assign(dow=hourly["timestamp"].dt.dayofweek, hour=hourly["timestamp"].dt.hour)
              .pivot_table(index="dow", columns="hour", values="speed_kmh", aggfunc="mean"))
grid.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
fig, ax = plt.subplots(figsize=(11, 3.5))
sns.heatmap(grid, cmap="RdYlGn", ax=ax, cbar_kws={"label": "mean speed (km/h)"})
ax.set_title("Mean speed by weekday and hour")
plt.tight_layout(); plt.savefig(OUT / "speed_heatmap.png", dpi=150); plt.show()

### B8. Weather coverage after the merge and neighbour feature coverage

In [ ]:
cols = ["temperature", "humidity", "precipitation", "neighbour_speed_t", "is_holiday"]
cov = hourly[cols].isna().sum().rename("nan_count")
cov.to_csv(OUT / "feature_nan_counts.csv")
print(cov)
print("Holiday hours:", int(hourly["is_holiday"].sum()), "| rainy hours:", int((hourly["precipitation"] > 0).sum()))

### B9. Summary table for the website

In [ ]:
summary = pd.DataFrame({
    "segments": [hourly["segment_id"].nunique()],
    "start": [hourly["timestamp"].min()],
    "end": [hourly["timestamp"].max()],
    "rows": [len(hourly)],
    "missing_rate": [round(float(hourly["speed_kmh"].isna().mean()), 4)],
})
summary.to_csv(OUT / "summary.csv", index=False)
summary